# In Class Assignment

**Name:** Christine Wu
**Date:** 04/09/26

## Submission Instructions
* Submit a link to your Jupyter Notebook in your GitHub repository. Make sure your settings will allow viewing of your notebook.

* Your notebook must be fully run (all outputs visible) and committed and pushed to your repository before the deadline.

## Activity

### 1. Load the credit card dataset and perform some initial EDA. In a markdown cell discuss some highlights from your EDA.

In [5]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 28.7 MB/s eta 0:00:00


In [8]:
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [2]:
# loading the dataset
credit = pd.read_csv('/Users/christinewu/Downloads/creditcard.csv')

# performing EDA with SweetViz
report = sv.analyze(credit)
report.show_html('credit_report.html')

                                             |          | [  0%]   00:00 -> (? left)

Report credit_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


Some highlights from my EDA:

1. The distribution of the limit_bal variable is skewed right, with most samples having less than 50k.
2. There is an uneven distribution of Sex, as more people in the dataset (60%) are Sex 2.
3. The most common value across all samples for the Education variable is 2.
4. The most common value across all samples for the Marriage variable is 2.
5. On average, individuals are about 35 years old.
6. There are various variables like Pays and Bills, but we can't assume what those mean since we do not have a data dictionary. However, it is worth noting that most Bill_Amounts are skewed right.

### 2. Prepare the data and build default tuned RandomForestClassifier and XGBClassifier models (use the AI-generated defaults for now). Do this with both the entire data set and using 5-fold cross-validation.  Calculate the mean metric score and standard deviation for the folds. In a markdown cell briefly discuss how your models performed. What does the mean and standard deviation across the folds tell you?

In [7]:
# 1. Define X and y
target_col = "default payment next month"

X = credit.drop(columns=[target_col])
y = credit[target_col]

# 2. Preprocessing
numeric_features = X.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features)
    ]
)

# 3. Define Models
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)


# 4. Pipelines
rf_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", rf_model)
])

xgb_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", xgb_model)
])

# 5. Full Dataset Evaluation
def full_data_score(pipeline, X, y):
    pipeline.fit(X, y) 
    y_prob = pipeline.predict_proba(X)[:, 1]  
    return roc_auc_score(y, y_prob)       

# 6. 5-fold CV
def cross_val_results(pipeline, X, y):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )
    return scores

# 7. Run everything
models = {
    "RandomForest": rf_pipeline,
    "XGBoost": xgb_pipeline
}

for name, model in models.items():
    full_score = full_data_score(model, X, y)
    cv_scores = cross_val_results(model, X, y)

    print(f"\n{name}")
    print(f"Full data ROC-AUC: {full_score:.4f}")
    print(f"CV mean ROC-AUC: {cv_scores.mean():.4f}")
    print(f"CV std ROC-AUC: {cv_scores.std():.4f}")
    print(f"Fold scores: {np.round(cv_scores, 4)}")


RandomForest
Full data ROC-AUC: 0.8595
CV mean ROC-AUC: 0.7787
CV std ROC-AUC: 0.0080
Fold scores: [0.7797 0.7849 0.7799 0.7854 0.7633]

XGBoost
Full data ROC-AUC: 0.8702
CV mean ROC-AUC: 0.7805
CV std ROC-AUC: 0.0081
Fold scores: [0.7853 0.7862 0.7833 0.7832 0.7646]


Both the RandomForest and XGBoost models performed well with ROC-AUC scores both being above 0.8. XGBoost performed slightly better as the ROC-AUC score is slightly higher. However, there is a lot of overfitting going on. The mean across the folds tells me that when I use the model, I would expect the average ROC-AUC score to be around 0.78. The standard deviation across the folds tells me how similar the scores will be across the folds. So since the standard deviations are very low, most folds have similar ROC-AUC scores and do not vary significantly. 

### 3. Look at the classification report for your two models. Does this change your evaluation of the models?

In [10]:
# Fit the models and print reports
for name, model in models.items():
    model.fit(X, y)
    y_pred = model.predict(X)

    print(f"\n{name}")
    print(classification_report(y, y_pred))


RandomForest
              precision    recall  f1-score   support

           0       0.90      0.87      0.88     18691
           1       0.58      0.66      0.62      5309

    accuracy                           0.82     24000
   macro avg       0.74      0.76      0.75     24000
weighted avg       0.83      0.82      0.82     24000


XGBoost
              precision    recall  f1-score   support

           0       0.86      0.96      0.91     18691
           1       0.76      0.43      0.55      5309

    accuracy                           0.84     24000
   macro avg       0.81      0.69      0.73     24000
weighted avg       0.83      0.84      0.83     24000



While Random Forest has a higher recall (more false positives), XGBoost has a higher precision (more confident predictions). Thus, RandomForest has a more balanced performance because XGBoost would miss many "defaulters".

### 4. Build your models using cross validation again, but this time for both use model features to adjust for the class imbalance. How did this impact model performance?

In [14]:
# 1. Compute class imbalance ratio

neg = (y == 0).sum()
pos = (y == 1).sum()

scale_pos_weight = neg / pos
print("scale_pos_weight:", scale_pos_weight)


# 2. Updated models with imbalance handling

rf_model_balanced = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight="balanced",   
    random_state=42,
    n_jobs=-1
)

xgb_model_balanced = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight, 
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)


# Pipelines
rf_pipeline_bal = Pipeline([
    ("prep", preprocessor),
    ("model", rf_model_balanced)
])

xgb_pipeline_bal = Pipeline([
    ("prep", preprocessor),
    ("model", xgb_model_balanced)
])


# 3. Cross-validation (ROC-AUC)
models_balanced = {
    "RandomForest_balanced": rf_pipeline_bal,
    "XGBoost_balanced": xgb_pipeline_bal
}

for name, model in models_balanced.items():
    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="roc_auc",
        n_jobs=-1
    )

    print(f"\n{name}")
    print(f"CV mean ROC-AUC: {scores.mean():.4f}")
    print(f"CV std ROC-AUC: {scores.std():.4f}")
    print(f"Fold scores: {np.round(scores, 4)}")


# 4. Cross-validated classification report (important)
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

for name, model in models_balanced.items():
    y_pred_cv = cross_val_predict(model, X, y, cv=5, n_jobs=-1)

    print(f"\n{name}")
    print(classification_report(y, y_pred_cv))

scale_pos_weight: 3.520625353173856

RandomForest_balanced
CV mean ROC-AUC: 0.7794
CV std ROC-AUC: 0.0092
Fold scores: [0.7737 0.7747 0.7946 0.7689 0.7849]

XGBoost_balanced
CV mean ROC-AUC: 0.7781
CV std ROC-AUC: 0.0063
Fold scores: [0.7719 0.7737 0.7893 0.7749 0.7806]

RandomForest_balanced
              precision    recall  f1-score   support

           0       0.87      0.85      0.86     18691
           1       0.52      0.57      0.54      5309

    accuracy                           0.79     24000
   macro avg       0.70      0.71      0.70     24000
weighted avg       0.79      0.79      0.79     24000


XGBoost_balanced
              precision    recall  f1-score   support

           0       0.88      0.81      0.84     18691
           1       0.48      0.62      0.54      5309

    accuracy                           0.77     24000
   macro avg       0.68      0.71      0.69     24000
weighted avg       0.79      0.77      0.78     24000



Though the ROC-AUC is mostly unchanged, RandomForest got worse at predicting class 1, but XGBoost improved and now catches more of the defaulters (compared to 43% from before). RandomForest is no longer the more balanced model.

### 5. Now try the XGBoost boostrapping (subsample =0.8) feature with. How did this affect performance?

In [13]:
# XGBoost with bootstrapping (row sampling)

xgb_bootstrap = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,             
    colsample_bytree=0.8, 
    scale_pos_weight=scale_pos_weight, 
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)

xgb_bootstrap_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", xgb_bootstrap)
])


# 1. Cross-validation ROC-AUC
scores = cross_val_score(
    xgb_bootstrap_pipeline,
    X,
    y,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

print("\nXGBoost (with subsample=0.8)")
print(f"CV mean ROC-AUC: {scores.mean():.4f}")
print(f"CV std ROC-AUC: {scores.std():.4f}")
print(f"Fold scores: {np.round(scores, 4)}")


# 2. Classification report (CV predictions)
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

y_pred_cv = cross_val_predict(
    xgb_bootstrap_pipeline,
    X,
    y,
    cv=5,
    n_jobs=-1
)

print("\nXGBoost (with subsample=0.8)")
print(classification_report(y, y_pred_cv))


XGBoost (with subsample=0.8)
CV mean ROC-AUC: 0.7798
CV std ROC-AUC: 0.0065
Fold scores: [0.7726 0.7757 0.7903 0.7764 0.784 ]

XGBoost (with subsample=0.8)
              precision    recall  f1-score   support

           0       0.88      0.81      0.84     18691
           1       0.48      0.61      0.54      5309

    accuracy                           0.77     24000
   macro avg       0.68      0.71      0.69     24000
weighted avg       0.79      0.77      0.78     24000



This gave the model a small regularization effect, since randomness was added to combat overfitting.

### 6. Do a brief tuning (2 or 3 features) for each model. It is your choice about whether to use the class imbalance adjustments or the subsample feature. Did your model performance increase or decrease? Do you think you chose the right parameters to tune?

In [15]:
# 1. Define X and y
target_col = "default payment next month"

X = credit.drop(columns=[target_col])
y = credit[target_col]


# 2. Preprocessing
numeric_features = X.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features)
    ]
)


# 3. Class imbalance ratio
neg = (y == 0).sum()
pos = (y == 1).sum()
scale_pos_weight = neg / pos


# 4. Base models
rf_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=scale_pos_weight,
        subsample=0.8,
        random_state=42,
        n_jobs=-1
    ))
])


# 5. Parameter grids
# Tuning 3 features for each model

rf_param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [8, 10, 12],
    "model__min_samples_leaf": [2, 4, 6]
}

xgb_param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [4, 5, 6],
    "model__learning_rate": [0.03, 0.05, 0.1]
}


# 6. Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# 7. Grid search for Random Forest
rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X, y)

print("\nRandom Forest Best Parameters:")
print(rf_grid.best_params_)

print("Random Forest Best CV ROC-AUC:")
print(round(rf_grid.best_score_, 4))


# 8. Grid search for XGBoost
xgb_grid = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=xgb_param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X, y)

print("\nXGBoost Best Parameters:")
print(xgb_grid.best_params_)

print("XGBoost Best CV ROC-AUC:")
print(round(xgb_grid.best_score_, 4))


# 9. Optional: show top results
rf_results = pd.DataFrame(rf_grid.cv_results_).sort_values(
    by="mean_test_score", ascending=False
)

xgb_results = pd.DataFrame(xgb_grid.cv_results_).sort_values(
    by="mean_test_score", ascending=False
)

print("\nTop 5 Random Forest tuning results:")
print(
    rf_results[
        ["params", "mean_test_score", "std_test_score", "rank_test_score"]
    ].head(5)
)

print("\nTop 5 XGBoost tuning results:")
print(
    xgb_results[
        ["params", "mean_test_score", "std_test_score", "rank_test_score"]
    ].head(5)
)


# 10. Save best models
best_rf_model = rf_grid.best_estimator_
best_xgb_model = xgb_grid.best_estimator_

Fitting 5 folds for each of 18 candidates, totalling 90 fits

Random Forest Best Parameters:
{'model__max_depth': 10, 'model__min_samples_leaf': 6, 'model__n_estimators': 300}
Random Forest Best CV ROC-AUC:
0.7792
Fitting 5 folds for each of 18 candidates, totalling 90 fits

XGBoost Best Parameters:
{'model__learning_rate': 0.03, 'model__max_depth': 4, 'model__n_estimators': 300}
XGBoost Best CV ROC-AUC:
0.7831

Top 5 Random Forest tuning results:
                                               params  mean_test_score  \
11  {'model__max_depth': 10, 'model__min_samples_l...         0.779218   
17  {'model__max_depth': 12, 'model__min_samples_l...         0.779159   
9   {'model__max_depth': 10, 'model__min_samples_l...         0.778806   
10  {'model__max_depth': 10, 'model__min_samples_l...         0.778618   
16  {'model__max_depth': 12, 'model__min_samples_l...         0.778525   

    std_test_score  rank_test_score  
11        0.008141                1  
17        0.008436         

For Random Forest, I tuned the number of trees, tree depth, and minimum samples per leaf. For XGBoost, I tuned the number of trees, tree depth, and learning rate. The best parameter combination for each model was selected based on the highest mean cross-validated ROC-AUC. Both of my model's performances increased, which tells me that I chose the right features to tune.

### 7. Which of your models was better out-of-the-box? Be specific.

Based on the tuning results, XGBoost performed better than Random Forest. The best cross-validated ROC AUC for XGBoost was approximately 0.7831, while the best Random Forest result was about 0.7792. This shows that after tuning, XGBoost achieved a higher average performance across folds. The difference is consistent across the top configurations. Both models have similar standard deviations, indicating comparable stability, but XGBoost maintains a slight advantage in predictive performance. Overall, XGBoost is the better model based on the tuned cross-validation results.